# 🏏 Rajasthan Royals — SupeRR Selector Season 2 (Round 1)
## Quantitative Framework: Most Reliable T20 Players Under Pressure (2022–2026)

**Author:** Ajay Mallick  
**Data Scope:** IPL & Men's T20 Internationals (2022–2026) | 807 Matches | 182,926 Legal Deliveries  
**Objective:** Build a reproducible, data-driven system to quantify how batters, bowlers, and all-rounders perform in high-pressure match situations compared with their baseline expectations.

---

### Methodological Highlights
1. **Dual Win Probability Engines:** Logistic Regression for 1st Innings; LightGBM for 2nd Innings, modeling run-chase probability dynamics.
2. **Win Probability Added Expectation (WPAE):** Benchmarking player contributions against context-adjusted peer baselines.
3. **Multi-Faceted Pressure Scenarios:** 9 specific scenarios (Scenarios A through I) capturing chasing pressure, collapse recovery, death overs, and tournament knockouts.
4. **Dynamic Recency & Shrinkage:** 12-month exponential half-life decay coupled with Empirical Bayes shrinkage ($C_{bat}, C_{bowl}$) to prevent small-sample distortions.
5. **Competition Rule Compliance:** 100% adherence to all competition rules, strictly excluding 68 Pakistan player IDs and associate-only players from the ranking pool.

## 1. Setup & Environment
Load core analysis packages and define paths to results and figures.

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

RES_DIR = 'outputs/results/'
FIG_DIR = 'outputs/figures/'

print("✓ Environment ready.")
print(f"✓ Results directory: {RES_DIR}")
print(f"✓ Figures directory: {FIG_DIR}")

## 2. Tournament Exclusions Audit

- **Pakistan / Pakistan A Exclusion:** All 68 `player_id`s mapped to `team_id` 25 (Pakistan) or 341 (Pakistan A) are strictly excluded from all player rankings (`excluded_pakistan_players.csv`). Their deliveries remain in the match corpus so Win Probability and peer baselines remain unbiased.
- **Associate-Only Exclusion:** Players whose appearances are exclusively for Associate nations without IPL or Full-Member experience are excluded from the eligible ranking pools.

In [ ]:
excl_pak = pd.read_csv(os.path.join(RES_DIR, 'excluded_pakistan_players.csv'))
print(f"Total Excluded Pakistan Players: {len(excl_pak)}")
excl_pak.head(10)

## 3. 🏆 Unified Top 10 Players Under Pressure

The overall leaderboard combining batting pressure score, bowling pressure score, 95% bootstrap confidence intervals, and probability of finishing in the top 10 ($P(\text{Top-10})$ across 300 iterations).

In [ ]:
top10_unified = pd.read_csv(os.path.join(RES_DIR, 'top10_unified.csv'))
# Verification: ensure zero excluded players exist in rankings
assert not top10_unified['player_id'].isin(excl_pak['player_id']).any(), "Exclusion check failed!"
top10_unified

## 4. 🏏 Top 10 Batters Under Pressure

*Eligibility: Full-Member / IPL player + ≥ 15 innings + ≥ 100 recency-weighted balls + ≥ 85 match-situation pressure balls overall.*

In [ ]:
top10_bat = pd.read_csv(os.path.join(RES_DIR, 'top10_batters_unstop.csv'))
top10_bat

### Phase-Specific Pressure Batters (≥ 85 Pressure Balls in Phase)
Evaluating specialists across Powerplay (Overs 1–6), Middle (Overs 7–15), and Death (Overs 16–20).

In [ ]:
phase_bat = pd.read_csv(os.path.join(RES_DIR, 'batter_phase_pressure_model_85b.csv'))
for phase in ['Powerplay', 'Middle', 'Death']:
    print(f"\n=== Top 5 in {phase} Overs ===")
    sub = phase_bat[phase_bat['phase'].str.lower() == phase.lower()].head(5)
    display(sub[['player_name', 'player_id', 'pressure_balls_phase', 'wpae_phase']])

## 5. 🎳 Top 10 Bowlers Under Pressure

*Eligibility: Full-Member / IPL player + ≥ 15 spells + ≥ 100 recency-weighted balls + ≥ 60 high-leverage balls.*

In [ ]:
top10_bowl = pd.read_csv(os.path.join(RES_DIR, 'top10_bowlers_unstop.csv'))
top10_bowl

## 6. Model Validation & Sensitivity Diagnostics

- **Out-of-Time (OOT) Testing:** Models trained on pre-2025 matches evaluated on 2025+ outcomes.
- **Split-Half Reliability:** Testing Spearman rank correlations across odd vs even matches.
- **Half-Life Sensitivity:** Testing recency half-life parameters (6, 12, 18, 24 months).

In [ ]:
with open(os.path.join(RES_DIR, 'validation_summary.json'), 'r') as f:
    val_summary = json.load(f)
print("Validation Summary:")
print(json.dumps(val_summary, indent=2))

sens_df = pd.read_csv(os.path.join(RES_DIR, 'sensitivity_analysis.csv'))
display(sens_df)

## 7. Visualizations & Analytical Charts
Displaying the primary diagnostic figures generated by the pipeline.

In [ ]:
figs_to_show = [
    ('rank_stability_unified.png', 'Bootstrap Rank Stability (Unified Top 10)'),
    ('calibration_inn2.png', 'Innings 2 Win Probability Model Calibration'),
    ('collapse_rebuild_curves.png', 'Collapse and Rebuild Probability Curves'),
    ('li_by_over.png', 'Leverage Index (LI) Distribution by Match Over'),
    ('usual_vs_pressure_scatter.png', 'Player Performance: Regular vs High Pressure Situations')
]

for filename, label in figs_to_show:
    img_path = os.path.join(FIG_DIR, filename)
    if os.path.exists(img_path):
        print(f"\n=== {label} ===")
        display(Image(filename=img_path, width=700))

## 8. Pipeline Reproduction Instructions

To reproduce the full pipeline end-to-end from scratch:
```bash
# 1. Install dependencies
pip install -r requirements.txt

# 2. Run sequential pipeline
python outputs/pipeline/s0_s1_scope_and_clean.py
python outputs/pipeline/s2_pressure.py
python outputs/pipeline/s3_wpae.py
python outputs/pipeline/s4_recency.py
python outputs/pipeline/s5_shrinkage.py
python outputs/pipeline/s6_validation.py
python outputs/pipeline/s7_report.py
python outputs/pipeline/s9_finalize.py
python outputs/pipeline/s9_finalize.py verify
```